[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/solutions/45_clip_contrastive_loss_solution.ipynb)

# 🟡 Solution: CLIP Contrastive Loss

Reference solution for symmetric image-text InfoNCE loss using stable logsumexp cross-entropy.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F


In [ ]:
# ✅ SOLUTION

def _l2_normalize(x: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return x / x.norm(dim=-1, keepdim=True).clamp_min(eps)


def _cross_entropy_from_logits(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    log_probs = logits - torch.logsumexp(logits, dim=-1, keepdim=True)
    return -log_probs[torch.arange(targets.numel(), device=targets.device), targets].mean()


def clip_contrastive_loss(image_embeds: torch.Tensor, text_embeds: torch.Tensor,
                          temperature: float = 0.07) -> torch.Tensor:
    if image_embeds.shape != text_embeds.shape:
        raise ValueError("image_embeds and text_embeds must have the same shape")

    image = _l2_normalize(image_embeds)
    text = _l2_normalize(text_embeds)
    logits = image @ text.T / temperature
    labels = torch.arange(image.shape[0], device=image.device)

    loss_i2t = _cross_entropy_from_logits(logits, labels)
    loss_t2i = _cross_entropy_from_logits(logits.T, labels)
    return 0.5 * (loss_i2t + loss_t2i)


In [ ]:
# Verify
torch.manual_seed(0)
image = torch.randn(4, 8)
text = torch.randn(4, 8)
print("Loss:", clip_contrastive_loss(image, text, temperature=0.2))

image_n = F.normalize(image, dim=-1)
text_n = F.normalize(text, dim=-1)
logits = image_n @ text_n.T / 0.2
labels = torch.arange(image.shape[0])
ref = 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels))
print("Ref: ", ref)


In [ ]:
# Run judge
from torch_judge import check
check('clip_contrastive_loss')
